# Introduction

In these exercises we'll apply groupwise analysis to our dataset.

Run the code cell below to load the data before running the exercises.

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 150

taster_names = ['Roger Voss', 'Paul Gregutt', "Kerin O'Keefe", 'Anna Lee C. Iijima',
                'Michael Schachner', 'Joe Czerwinski']
taster_handles = ['@vossroger', '@paulgwine', '@kerinokeefe', '@annaiijima',
                  '@wineschach', '@JoeCz']

country_pool = ['Italy', 'Portugal', 'US', 'Spain', 'France', 'Australia']
variety_pool = ['Pinot Noir', 'Chardonnay', 'Cabernet Sauvignon', 'Riesling',
                'Sangiovese', 'Shiraz', 'Tempranillo', 'White Blend']

countries = list(np.random.choice(country_pool, n))
varieties = list(np.random.choice(variety_pool, n))
points = list(np.random.randint(82, 98, n))
prices = [round(float(p), 1) for p in np.random.uniform(10, 200, n)]

taster_idx = [i % 6 for i in range(n)]
taster_name_col = [None if i % 7 == 0 else taster_names[taster_idx[i]] for i in range(n)]
taster_handle_col = [None if i % 7 == 0 else taster_handles[taster_idx[i]] for i in range(n)]

reviews = pd.DataFrame({
    'country': countries,
    'description': [f'Wine description for review {i}.' for i in range(n)],
    'designation': [None if i % 5 == 0 else f'Label {i % 15}' for i in range(n)],
    'points': points,
    'price': prices,
    'province': [f'Province {i % 8}' for i in range(n)],
    'region_1': [None if i % 3 == 0 else f'Region {i % 8}' for i in range(n)],
    'region_2': [None if i % 4 == 0 else f'Sub-region {i % 5}' for i in range(n)],
    'taster_name': taster_name_col,
    'taster_twitter_handle': taster_handle_col,
    'title': [f'Winery {i % 20} {2010 + i % 10} Vintage Wine' for i in range(n)],
    'variety': varieties,
    'winery': [f'Winery {i % 20}' for i in range(n)],
})

pd.set_option('display.max_rows', 5)
print(f'Loaded {len(reviews)} wine reviews with {reviews.shape[1]} columns.')
print('Setup complete.')
reviews.head()

Loaded 150 wine reviews with 13 columns.
Setup complete.


,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,Spain,Wine description for review 0.,NaN,92,80.8,Province 0,NaN,NaN,NaN,NaN,Winery 0 2010 Vintage Wine,Sangiovese,Winery 0
1,France,Wine description for review 1.,Label 1,93,85.0,Province 1,Region 1,Sub-region 1,Paul Gregutt,@paulgwine,Winery 1 2011 Vintage Wine,Tempranillo,Winery 1
2,US,Wine description for review 2.,Label 2,90,170.4,Province 2,Region 2,Sub-region 2,Kerin O'Keefe,@kerinokeefe,Winery 2 2012 Vintage Wine,Tempranillo,Winery 2
3,France,Wine description for review 3.,Label 3,90,186.7,Province 3,NaN,Sub-region 3,Anna Lee C. Iijima,@annaiijima,Winery 3 2013 Vintage Wine,Sangiovese,Winery 3
4,France,Wine description for review 4.,Label 4,83,23.4,Province 4,Region 4,NaN,Michael Schachner,@wineschach,Winery 4 2014 Vintage Wine,Tempranillo,Winery 4


# Exercises

## 1.
Who are the most common wine reviewers in the dataset? Create a `Series` whose index is the `taster_twitter_handle` category from the dataset, and whose values count how many reviews each person wrote.

In [2]:
reviews_written = reviews.groupby('taster_twitter_handle').size()
reviews_written

taster_twitter_handle
@JoeCz         22
@annaiijima    21
               ..
@vossroger     21
@wineschach    22
Length: 6, dtype: int64

**Alternative:**
```python
reviews_written = reviews.groupby('taster_twitter_handle').taster_twitter_handle.count()
```

**Key insight:** `.groupby().size()` counts **all rows** in each group (including NaN values in other columns). `.groupby().count()` counts **non-NaN values** in a specific column. For row counting, prefer `.size()` — it's clearer in intent and slightly faster.

## 2.
What is the best wine I can buy for a given amount of money? Create a `Series` whose index is wine prices and whose values is the maximum number of points a wine costing that much was given in a review. Sort the values by price, ascending.

In [3]:
best_rating_per_price = reviews.groupby('price').points.max().sort_index()
best_rating_per_price

price
11.0     82
12.9     87
         ..
196.1    97
196.5    97
Name: points, Length: 141, dtype: int32

**Key insight:** After a `.groupby()` operation, the grouped-by column becomes the **index** of the result. `.sort_index()` sorts by that index (price here) while `.sort_values()` would sort by the aggregated values (points). Knowing which axis to sort is key.

## 3.
What are the minimum and maximum prices for each `variety` of wine? Create a `DataFrame` whose index is the `variety` category from the dataset and whose values are the `min` and `max` values thereof.

In [4]:
price_extremes = reviews.groupby('variety').price.agg(['min', 'max'])
price_extremes

,min,max
variety,,
Cabernet Sauvignon,17.4,196.1
Chardonnay,13.3,187.4
...,...,...
Tempranillo,19.1,174.8
White Blend,19.1,189.4


**Key insight:** Pass a **list of strings** to `.agg()` to apply multiple aggregations at once — the result is a DataFrame with one column per function. Use `['min', 'max']` (strings), not `[min, max]` (Python builtins), to avoid FutureWarning deprecation in modern pandas.

## 4.
What are the most expensive wine varieties? Create a variable `sorted_varieties` containing a copy of the dataframe from the previous question where varieties are sorted in descending order based on minimum price, then on maximum price (to break ties).

In [5]:
sorted_varieties = price_extremes.sort_values(by=['min', 'max'], ascending=False)
sorted_varieties

,min,max
variety,,
White Blend,19.1,189.4
Tempranillo,19.1,174.8
...,...,...
Shiraz,12.9,189.3
Sangiovese,11.0,186.7


**Key insight:** `sort_values(by=['col1', 'col2'])` sorts by `col1` first; rows with equal `col1` values are then ordered by `col2`. This multi-key sort lets you define a complete tiebreaker ordering — a common pattern after `.agg()` to rank groups.

## 5.
Create a `Series` whose index is reviewers and whose values is the average review score given out by that reviewer. Hint: you will need the `taster_name` and `points` columns.

In [6]:
reviewer_mean_ratings = reviews.groupby('taster_name').points.mean()
reviewer_mean_ratings

taster_name
Anna Lee C. Iijima    88.714286
Joe Czerwinski        89.727273
                        ...    
Paul Gregutt          87.666667
Roger Voss            89.952381
Name: points, Length: 6, dtype: float64

**Key insight:** Chaining `.groupby(key).column.agg()` is the standard pattern: group by one column, aggregate a different column. The grouped key becomes the index; the aggregated column becomes the values.

Are there significant differences in the average scores assigned by the various reviewers? Run the cell below to use the `describe()` method to see a summary of the range of values.

In [7]:
reviewer_mean_ratings.describe()

count     6.000000
mean     89.264430
           ...    
75%      89.896104
max      89.954545
Name: points, Length: 8, dtype: float64

## 6.
What combination of countries and varieties are most common? Create a `Series` whose index is a `MultiIndex` of `{country, variety}` pairs. For example, a Pinot Noir produced in the US should map to `{"US", "Pinot Noir"}`. Sort the values in the `Series` in descending order based on wine count.

In [8]:
country_variety_counts = reviews.groupby(['country', 'variety']).size().sort_values(ascending=False)
country_variety_counts

country    variety           
Australia  Pinot Noir            6
US         Cabernet Sauvignon    6
                                ..
Italy      Sangiovese            1
US         Pinot Noir            1
Length: 46, dtype: int64

**Key insight:** Passing a **list of columns** to `.groupby()` creates a **MultiIndex** — a hierarchical index with one level per column. This lets you count or aggregate over combinations of categorical variables. The resulting index has tuples as labels: `(country, variety)`.

---
## Session Summary — Day 4 Pandas

| Exercise | Topic | Key Concept |
|----------|-------|-------------|
| 1 | Count reviews per taster | `.groupby().size()` — counts all rows; prefer over `.count()` for row totals |
| 2 | Best rating per price | `.groupby().max().sort_index()` — sort by **index** (the groupby key), not values |
| 3 | Price range per variety | `.agg(['min', 'max'])` — use string names, not Python builtins, to avoid FutureWarning |
| 4 | Sort by multiple columns | `.sort_values(by=['col1', 'col2'])` — first key sorts primary, second breaks ties |
| 5 | Mean score per reviewer | `.groupby(key).column.mean()` — standard group-then-aggregate chain |
| 6 | Multi-key groupby | `.groupby(['col1', 'col2'])` — creates a MultiIndex; index labels are `(val1, val2)` tuples |